In [3]:
!pip install mediapipe tensorflow scikit-learn numpy pandas matplotlib seaborn opencv-python-headless


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import os
import cv2
import time
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python as mp_tasks
from mediapipe.tasks.python import vision

# DYNAMIC_GESTURES = ['Z', 'J']
DYNAMIC_GESTURES = ['kolo', 'Z', 'J']
SEQUENCE_LENGTH = 40
SAMPLES_PER_GESTURE = 400
DATA_DIR = 'dynamic_dataset'

os.makedirs(DATA_DIR, exist_ok=True)
for gesture in DYNAMIC_GESTURES:
    os.makedirs(os.path.join(DATA_DIR, gesture), exist_ok=True)

def extract_landmarks_normalized(hand_landmarks):
    base_x, base_y, base_z = hand_landmarks[0].x, hand_landmarks[0].y, hand_landmarks[0].z
    max_dist = max(
        np.sqrt((lm.x - base_x)**2 + (lm.y - base_y)**2)
        for lm in hand_landmarks
    )
    scale = max_dist if max_dist > 0 else 1.0

    vector = []
    for lm in hand_landmarks:
        vector.extend([
            (lm.x - base_x) / scale,
            (lm.y - base_y) / scale,
            (lm.z - base_z) / scale,
        ])
    return vector

def collect_dynamic_dataset():
    base_options = mp_tasks.BaseOptions(model_asset_path='../hand_landmarker.task')
    options = vision.HandLandmarkerOptions(
        base_options=base_options,
        num_hands=1,
        min_hand_detection_confidence=0.5
    )
    detector = vision.HandLandmarker.create_from_options(options)
    cap = cv2.VideoCapture(0)

    for gesture in DYNAMIC_GESTURES:
        gesture_dir = os.path.join(DATA_DIR, gesture)
        existing = len(os.listdir(gesture_dir))
        samples_needed = SAMPLES_PER_GESTURE - existing
        
        if samples_needed <= 0:
            print(f"[SKIP] {gesture}: {existing} samples already exist")
            continue

        print(f"\n{'='*50}")
        print(f"Collecting gesture: [{gesture}] — {samples_needed} samples needed")
        print("Press SPACE to start, Q to quit")
        print(f"{'='*50}")

        sample_idx = existing

        while sample_idx < SAMPLES_PER_GESTURE:
            while True:
                ret, frame = cap.read()
                if not ret: break
                frame = cv2.flip(frame, 1)
                cv2.putText(frame,
                    f"Gesture [{gesture}] | Collected: {sample_idx}/{SAMPLES_PER_GESTURE}",
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2)
                cv2.putText(frame,
                    "Press SPACE to record this gesture",
                    (10, 70), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
                cv2.imshow('Dataset Collection', frame)
                key = cv2.waitKey(10) & 0xFF
                if key == ord(' '):
                    break
                if key == ord('q'):
                    cap.release()
                    cv2.destroyAllWindows()
                    return

            for countdown in range(0):
                ret, frame = cap.read()
                frame = cv2.flip(frame, 1)
                cv2.putText(frame, f"Starting in {countdown}...",
                    (10, 150), cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 0, 255), 3)
                cv2.imshow('Dataset Collection', frame)
                cv2.waitKey(1000)

            sequence = []
            frames_recorded = 0

            while frames_recorded < SEQUENCE_LENGTH:
                ret, frame = cap.read()
                if not ret: break
                frame = cv2.flip(frame, 1)
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                mp_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
                result = detector.detect(mp_img)

                if result.hand_landmarks:
                    landmarks = extract_landmarks_normalized(result.hand_landmarks[0])
                else:
                    landmarks = [0.0] * 63

                sequence.append(landmarks)
                frames_recorded += 1

                progress = int((frames_recorded / SEQUENCE_LENGTH) * frame.shape[1])
                cv2.rectangle(frame, (0, 0), (progress, 15), (0, 255, 0), -1)
                cv2.putText(frame, f"RECORDING [{gesture}] {frames_recorded}/{SEQUENCE_LENGTH}",
                    (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
                cv2.imshow('Dataset Collection', frame)
                cv2.waitKey(1)

            if len(sequence) == SEQUENCE_LENGTH:
                seq_array = np.array(sequence, dtype=np.float32)
                save_path = os.path.join(gesture_dir, f"{sample_idx:04d}.npy")
                np.save(save_path, seq_array)
                sample_idx += 1
                print(f"  Saved sample {sample_idx}/{SAMPLES_PER_GESTURE}")
            else:
                print("  Sample discarded (incomplete sequence)")

    cap.release()
    cv2.destroyAllWindows()
    print("\n Dataset collection complete!")

if __name__ == "__main__":
    collect_dynamic_dataset()


Press SPACE to start, Q to quit
  Saved sample 301/400
  Saved sample 302/400
  Saved sample 303/400
  Saved sample 304/400
  Saved sample 305/400
  Saved sample 306/400
  Saved sample 307/400
